# DART 재무데이터 수집·변환 품질 점검 v1

매출 예측·밸류에이션 없이 **DART 데이터가 표준 필드로 제대로 수집·분기화되는지만** 검증하는 노트북.

- Cell 1 : 설정 (점검 대상 종목)
- Cell 2 : DART long → 표준 wide 변환 모듈 (FCFF v1.1 과 동일)
- Cell 3 : 계정 매핑 진단 (`check_field_mapping`, `diagnose_accounts`)
- Cell 4 : 품질 점검 실행 — 종목별 리포트 + 일괄 요약 매트릭스

**보는 법**
1. 커버리지 ❌ 필드 → Cell 3 진단으로 실제 계정명 확인 후 Cell 2 FIELD_MAP 에 패턴 추가
2. `sum≈FY` MISMATCH / 음수 분기 → 누적·3개월 오감지 또는 정정공시 혼입 신호
3. 최근 8분기 테이블에서 매출·영업이익 규모가 실제(공시)와 맞는지 눈검사

In [1]:
# ═══════════════════════════════════════════════════════════════
#  ★ 입력 변수 — 이 셀만 수정하세요
# ═══════════════════════════════════════════════════════════════
# 점검 대상 (6자리 코드, 'A' 접두어 허용)
CHECK_TICKERS = [
    "005930",   # 삼성전자
    "000660",   # SK하이닉스
    "035420",   # NAVER
    "005380",   # 현대차
    "051910",   # LG화학
]

DB_INFO = {
    "host": "192.168.0.230",
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
TABLE_DART_FS = "korea_fs_data_from_DART_V2"

VERBOSE = True
print("[OK] 설정 완료 —", CHECK_TICKERS)


[OK] 설정 완료 — ['005930', '000660', '035420', '005380', '051910']


## Cell 2 · DART → 표준 wide 변환

In [2]:
# ═══════════════════════════════════════════════════════════════
#  DART long → 표준 wide 변환 모듈
#  - account_id 우선 매칭, 실패 시 account_nm 정규식 fallback
#  - IS/CIS: 누적 vs 3개월 자동 감지 후 분기화 (Q4 = FY − 3개분기)
#  - CF: 항상 누적으로 간주 → 차분 (Q2=H1−Q1, Q3=Q3−H1, Q4=FY−Q3)
#  - BS: 시점 잔액 그대로
# ═══════════════════════════════════════════════════════════════
import re
import numpy as np
import pandas as pd
import pymysql
from datetime import datetime


def log(tag, msg):
    print(f"[{datetime.now():%H:%M:%S}][{tag}] {msg}", flush=True)


def norm_ticker(t: str) -> str:
    """'A005930' → '005930'"""
    t = str(t).strip().upper()
    return t[1:].zfill(6) if t.startswith("A") else t.zfill(6)


def to_dg_ticker(t: str) -> str:
    """'005930' → 'A005930' (forecast 테이블용)"""
    return "A" + norm_ticker(t)


# ───────────────────────────────────────────────────────────────
#  표준 필드 매핑 정의
#    ids : account_id 후보 (정확 일치, 접두어 ifrs_/ifrs-full_ 모두 등록)
#    nm  : account_nm 정규식 후보 (앞에 있을수록 우선)
#    sj  : 허용 재무제표 (앞에 있을수록 우선; IS 우선, 없으면 CIS)
#    agg : 'pick'=대표 계정 1개 선택(중복합산 방지) / 'sum'=매칭 계정 전부 합산
# ───────────────────────────────────────────────────────────────
def _ids(*stems):
    out = []
    for s in stems:
        out += [f"ifrs_{s}", f"ifrs-full_{s}"]
    return out


FIELD_MAP = {
    # ── 손익 (flow) ──
    "revenue": dict(
        ids=_ids("Revenue") + ["dart_Revenue"],
        nm=[r"^매출액$", r"^매출$", r"^수익\(매출액\)$", r"^영업수익$"],
        sj=["IS", "CIS"], agg="pick"),
    "operating_income": dict(
        ids=["dart_OperatingIncomeLoss"] + _ids("OperatingIncomeLoss"),
        nm=[r"^영업이익", r"^영업손익"],
        sj=["IS", "CIS"], agg="pick"),
    "pretax_income": dict(
        ids=_ids("ProfitLossBeforeTax"),
        nm=[r"법인세비용차감전", r"^세전.*이익"],
        sj=["IS", "CIS"], agg="pick"),
    "tax_expense": dict(
        ids=_ids("IncomeTaxExpenseContinuingOperations", "IncomeTaxExpense"),
        nm=[r"^법인세비용"],
        sj=["IS", "CIS"], agg="pick"),
    "interest_expense": dict(
        ids=_ids("FinanceCosts") + ["dart_InterestExpenseFinanceExpense"],
        nm=[r"^이자비용", r"^금융비용"],
        sj=["IS", "CIS"], agg="pick"),

    # ── 현금흐름 (flow, 누적) ──
    "da_cf": dict(
        ids=["dart_AdjustmentsForDepreciationExpense"] +
            _ids("AdjustmentsForDepreciationExpense",
                 "AdjustmentsForDepreciationAndAmortisationExpense",
                 "DepreciationAndAmortisationExpense"),
        nm=[r"감가상각비와\s*무형자산상각", r"감가상각비\s*및\s*상각",
            r"감가상각"],                    # 앞머리 고정 제거 — "유형자산 감가상각비" 등 대응
        sj=["CF"], agg="pick"),
    "intangible_amort_cf": dict(
        ids=["dart_AmortisationExpense"] +
            _ids("AdjustmentsForAmortisationExpense", "AmortisationExpense"),
        nm=[r"^(?!.*감가상각).*무형자산\s*상각"],   # 합산계정(감가상각비와 무형자산상각비) 제외 — da_cf 와 이중계상 방지
        sj=["CF"], agg="pick"),
    "capex_tangible": dict(
        ids=_ids("PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
                 "PurchaseOfPropertyPlantAndEquipment"),
        nm=[r"유형자산의\s*취득", r"유형자산의\s*증가", r"유형자산\s*취득",
            r"토지.*취득|건설중인자산.*(취득|증가)"],
        sj=["CF"], agg="pick"),
    "capex_intangible": dict(
        ids=_ids("PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities",
                 "PurchaseOfIntangibleAssets"),
        nm=[r"무형자산의\s*취득", r"무형자산의\s*증가", r"무형자산\s*취득"],
        sj=["CF"], agg="pick"),

    # ── 재무상태 (stock) ──
    "receivables": dict(
        ids=_ids("TradeAndOtherCurrentReceivables", "CurrentTradeReceivables"),
        nm=[r"^매출채권$", r"^매출채권\s*및", r"^매출채권과"],
        sj=["BS"], agg="pick"),
    "inventories": dict(
        ids=_ids("Inventories"),
        nm=[r"^재고자산"],
        sj=["BS"], agg="pick"),
    "prepaid_expenses": dict(
        ids=[], nm=[r"^선급비용"], sj=["BS"], agg="pick"),
    "payables": dict(
        ids=_ids("TradeAndOtherCurrentPayables", "CurrentTradePayables"),
        nm=[r"^매입채무$", r"^매입채무\s*및", r"^매입채무와"],
        sj=["BS"], agg="pick"),
    "accrued_expenses": dict(
        ids=[], nm=[r"^미지급비용"], sj=["BS"], agg="pick"),
    "other_payables": dict(
        ids=[], nm=[r"^미지급금"], sj=["BS"], agg="pick"),
    "advances_received": dict(
        ids=[], nm=[r"^선수금"], sj=["BS"], agg="pick"),
    "contract_liabilities": dict(
        ids=_ids("ContractLiabilities"),
        nm=[r"^계약부채"], sj=["BS"], agg="pick"),

    "short_term_debt": dict(
        ids=["dart_ShortTermBorrowings"] + _ids("ShorttermBorrowings"),
        nm=[r"^단기차입금"], sj=["BS"], agg="pick"),
    "current_lt_debt": dict(
        ids=[], nm=[r"^유동성장기부채", r"^유동성장기차입금", r"^유동성사채"],
        sj=["BS"], agg="sum"),
    "bonds": dict(
        ids=[], nm=[r"^사채$", r"^사채\("], sj=["BS"], agg="pick"),
    "long_term_debt": dict(
        ids=["dart_LongTermBorrowingsGross"],
        nm=[r"^장기차입금"], sj=["BS"], agg="pick"),
    "lease_liab": dict(
        ids=_ids("LeaseLiabilities", "CurrentLeaseLiabilities",
                 "NoncurrentLeaseLiabilities"),
        nm=[r"리스부채"], sj=["BS"], agg="sum"),

    "cash": dict(
        ids=_ids("CashAndCashEquivalents"),
        nm=[r"^현금및현금성자산"], sj=["BS"], agg="pick"),
    "short_term_invest": dict(
        ids=["dart_ShortTermDepositsNotClassifiedAsCashEquivalents"],
        nm=[r"^단기금융상품", r"^단기투자자산"], sj=["BS"], agg="pick"),
    "total_equity": dict(
        ids=_ids("Equity"),
        nm=[r"^자본총계"], sj=["BS"], agg="pick"),
    "total_assets": dict(
        ids=_ids("Assets"),
        nm=[r"^자산총계"], sj=["BS"], agg="pick"),
}

QUARTER_ORDER = {"Q1": 1, "H1": 2, "Q3": 3, "FY": 4}
FLOW_SJ  = {"IS", "CIS", "CF"}


def _load_ticker_long(ticker: str, db_info: dict, table: str) -> pd.DataFrame:
    conn = pymysql.connect(**db_info, charset="utf8mb4")
    try:
        df = pd.read_sql(
            f"""SELECT bsns_year, quarter, sj_div, account_id, account_nm,
                       thstrm_amount, report_date
                FROM {table} WHERE ticker = %s""",
            conn, params=[norm_ticker(ticker)])
    finally:
        conn.close()
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    df["report_date"] = pd.to_datetime(df["report_date"])
    return df


def _match_field(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    """필드 정의(spec)에 맞는 행 선택. 반환: (bsns_year, quarter) 별 단일 값."""
    sub = df[df["sj_div"].isin(spec["sj"])].copy()
    if sub.empty:
        return pd.DataFrame()

    # 1) account_id 정확 일치
    hit = sub[sub["account_id"].isin(spec["ids"])] if spec["ids"] else pd.DataFrame()

    # 2) 실패 시 account_nm 정규식 (패턴 순서 = 우선순위)
    if hit.empty and spec["nm"]:
        for pat in spec["nm"]:
            m = sub[sub["account_nm"].astype(str).str.strip()
                       .str.contains(pat, regex=True, na=False)]
            if not m.empty:
                hit = m
                break
    if hit.empty:
        return pd.DataFrame()

    # sj_div 우선순위 (IS > CIS 등): 상위 sj에 데이터가 있으면 그것만
    for sj in spec["sj"]:
        h2 = hit[hit["sj_div"] == sj]
        if not h2.empty:
            hit = h2
            break

    if spec["agg"] == "sum":
        # 서로 다른 account_id 를 (연도,분기)별 합산 (리스부채 유동+비유동 등)
        out = (hit.groupby(["bsns_year", "quarter"], as_index=False)
                  ["thstrm_amount"].sum(min_count=1))
    else:
        # 'pick': 기간 커버리지가 가장 넓은 대표 account_id 1개만 사용 (중복합산 방지)
        cov = hit.groupby("account_id")["bsns_year"].count().sort_values(ascending=False)
        best = cov.index[0]
        out = hit[hit["account_id"] == best][
            ["bsns_year", "quarter", "thstrm_amount"]].copy()
        # 같은 (연도,분기) 중복 시 첫 행
        out = out.drop_duplicates(subset=["bsns_year", "quarter"])
    return out


def _detect_cumulative(pivot: pd.DataFrame) -> bool:
    """
    IS 계열 flow 가 누적인지 3개월치인지 자동 감지.
    (Q1+H1+Q3)/FY 중앙값: 3개월치 ≈ 0.75, 누적 ≈ 1.5 → 임계 1.1
    """
    ratios = []
    for y, row in pivot.iterrows():
        if all(pd.notnull(row.get(q)) for q in ("Q1", "H1", "Q3", "FY")) \
                and row["FY"] not in (0, None):
            ratios.append((row["Q1"] + row["H1"] + row["Q3"]) / row["FY"])
    if not ratios:
        return False   # 판단 불가 → 3개월치 가정 (보수적)
    return float(np.median(ratios)) > 1.1


def _flow_to_quarterly(series_df: pd.DataFrame, force_cumulative: bool = None):
    """
    flow 항목 (연도,분기,값) → 분기화 값 dict {(year,'Qn'): value}.
    force_cumulative: None=자동감지, True=누적 차분, False=3개월치 취급
    반환: (dict, cumulative여부)
    """
    pivot = series_df.pivot_table(index="bsns_year", columns="quarter",
                                  values="thstrm_amount", aggfunc="first")
    cum = _detect_cumulative(pivot) if force_cumulative is None else force_cumulative

    out = {}
    for y, row in pivot.iterrows():
        q1, h1, q3, fy = (row.get("Q1"), row.get("H1"),
                          row.get("Q3"), row.get("FY"))
        if cum:
            out[(y, "Q1")] = q1
            out[(y, "Q2")] = h1 - q1 if pd.notnull(h1) and pd.notnull(q1) else np.nan
            out[(y, "Q3")] = q3 - h1 if pd.notnull(q3) and pd.notnull(h1) else np.nan
            out[(y, "Q4")] = fy - q3 if pd.notnull(fy) and pd.notnull(q3) else np.nan
        else:
            out[(y, "Q1")] = q1
            out[(y, "Q2")] = h1
            out[(y, "Q3")] = q3
            if all(pd.notnull(v) for v in (fy, q1, h1, q3)):
                out[(y, "Q4")] = fy - (q1 + h1 + q3)
            else:
                out[(y, "Q4")] = np.nan
    return out, cum


_QDATE = {"Q1": "-03-31", "Q2": "-06-30", "Q3": "-09-30", "Q4": "-12-31"}


def load_dart_financials_wide(ticker: str, db_info: dict,
                              table_name: str = None,
                              item_keys=None, fillna_zero: bool = False,
                              verbose: bool = False) -> pd.DataFrame:
    """
    DART long 테이블 → 분기 wide DataFrame (index=분기말 date, 단위=원).
    기존 load_korea_financials_wide 와 동일한 사용 패턴.
    """
    table_name = table_name or TABLE_DART_FS
    raw = _load_ticker_long(ticker, db_info, table_name)
    if raw.empty:
        return pd.DataFrame()

    fields = item_keys or list(FIELD_MAP.keys())
    col_data, cum_info = {}, {}

    for f in fields:
        spec = FIELD_MAP[f]
        sel = _match_field(raw, spec)
        if sel.empty:
            continue
        if spec["sj"][0] in FLOW_SJ:
            force = True if spec["sj"] == ["CF"] else None   # CF는 항상 누적
            qvals, cum = _flow_to_quarterly(sel, force_cumulative=force)
            cum_info[f] = cum
        else:  # BS: 시점 잔액, H1→Q2 라벨만 변경
            qvals = {}
            for _, r in sel.iterrows():
                q = {"Q1": "Q1", "H1": "Q2", "Q3": "Q3", "FY": "Q4"}[r["quarter"]]
                qvals[(int(r["bsns_year"]), q)] = r["thstrm_amount"]
        col_data[f] = qvals

    if not col_data:
        return pd.DataFrame()

    all_keys = sorted({k for v in col_data.values() for k in v})
    idx = pd.to_datetime([f"{y}{_QDATE[q]}" for y, q in all_keys])
    wide = pd.DataFrame(
        {f: [col_data[f].get(k, np.nan) for k in all_keys] for f in col_data},
        index=idx).sort_index()

    if fillna_zero:
        wide = wide.fillna(0.0)

    if verbose:
        cum_flows = [f for f, c in cum_info.items() if c]
        log(norm_ticker(ticker),
            f"wide shape={wide.shape} 기간={wide.index.min().date()}~{wide.index.max().date()}"
            + (f"  누적차분 적용: {cum_flows}" if cum_flows else ""))
    return wide


print("[OK] DART wide 변환 모듈 로드 완료")


[OK] DART wide 변환 모듈 로드 완료


## Cell 3 · 계정 매핑 진단

In [3]:
# ═══════════════════════════════════════════════════════════════
#  진단 — 특정 종목의 DART 계정 목록 덤프 (매핑 보강용)
#  da=0 처럼 필드가 비면 이 셀을 실행해 실제 account_id/account_nm 을 확인하고
#  Cell 2 의 FIELD_MAP 에 패턴을 추가하세요.
# ═══════════════════════════════════════════════════════════════
import pandas as pd


def diagnose_accounts(ticker: str, sj_div: str = None, keyword: str = None,
                      top: int = 60):
    """
    종목의 (sj_div, account_id, account_nm) 별 커버리지·최근 FY 금액 요약.
    sj_div : 'CF','BS','IS','CIS' 필터 (None=전체)
    keyword: account_nm 포함 검색어 (예: '상각', '취득', '차입')
    """
    raw = _load_ticker_long(ticker, DB_INFO, TABLE_DART_FS)
    if raw.empty:
        print(f"❌ {ticker}: 데이터 없음")
        return None
    df = raw.copy()
    if sj_div:
        df = df[df["sj_div"] == sj_div]
    if keyword:
        df = df[df["account_nm"].astype(str).str.contains(keyword, na=False)]

    last_fy = df[df["quarter"] == "FY"]["bsns_year"].max()
    fy_amt = (df[(df["quarter"] == "FY") & (df["bsns_year"] == last_fy)]
              .set_index(["sj_div", "account_id", "account_nm"])["thstrm_amount"])

    g = (df.groupby(["sj_div", "account_id", "account_nm"])
           .agg(n_periods=("bsns_year", "count"),
                yr_min=("bsns_year", "min"), yr_max=("bsns_year", "max"))
           .sort_values("n_periods", ascending=False))
    g["last_FY_억원"] = (fy_amt.reindex(g.index) / 1e8).round(0)
    print(f"[{norm_ticker(ticker)}] sj={sj_div or 'ALL'} kw={keyword or '-'} "
          f"— 계정 {len(g)}개 (최근 FY={last_fy})")
    with pd.option_context("display.max_rows", top, "display.width", 200,
                           "display.max_colwidth", 45):
        print(g.head(top).to_string())
    return g


def check_field_mapping(ticker: str):
    """FIELD_MAP 각 필드가 이 종목에서 어떤 계정으로 resolve 되는지 일람."""
    raw = _load_ticker_long(ticker, DB_INFO, TABLE_DART_FS)
    print(f"[{norm_ticker(ticker)}] 필드 → 매칭 계정")
    for f, spec in FIELD_MAP.items():
        sel_rows = raw[raw["sj_div"].isin(spec["sj"])]
        hit = sel_rows[sel_rows["account_id"].isin(spec["ids"])] if spec["ids"] else sel_rows.iloc[0:0]
        via = "id"
        if hit.empty and spec["nm"]:
            for pat in spec["nm"]:
                m = sel_rows[sel_rows["account_nm"].astype(str).str.strip()
                             .str.contains(pat, regex=True, na=False)]
                if not m.empty:
                    hit, via = m, f"nm:{pat}"
                    break
        if hit.empty:
            print(f"  ❌ {f:<22} 매칭 없음")
        else:
            names = hit.groupby(["account_id", "account_nm"]).size() \
                       .sort_values(ascending=False)
            best = names.index[0]
            print(f"  ✅ {f:<22} [{via}] {best[0]} / {best[1]} "
                  f"({names.iloc[0]}기간{', 후보 '+str(len(names))+'개' if len(names)>1 else ''})")


# 실행 예: D&A가 0으로 나온 종목의 CF 계정 확인
check_field_mapping(CHECK_TICKERS[0])
print()
diagnose_accounts(CHECK_TICKERS[0], sj_div="CF", keyword="상각")


C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


[005930] 필드 → 매칭 계정
  ✅ revenue                [id] ifrs-full_Revenue / 수익(매출액) (17기간, 후보 4개)
  ✅ operating_income       [id] dart_OperatingIncomeLoss / 영업이익 (23기간, 후보 2개)
  ✅ pretax_income          [id] ifrs-full_ProfitLossBeforeTax / 법인세비용차감전순이익(손실) (20기간, 후보 3개)
  ✅ tax_expense            [id] ifrs-full_IncomeTaxExpenseContinuingOperations / 법인세비용 (19기간, 후보 3개)
  ✅ interest_expense       [id] ifrs-full_FinanceCosts / 금융비용 (9기간, 후보 2개)
  ❌ da_cf                  매칭 없음
  ❌ intangible_amort_cf    매칭 없음
  ✅ capex_tangible         [id] ifrs-full_PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities / 유형자산의 취득 (25기간, 후보 2개)
  ✅ capex_intangible       [id] ifrs-full_PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities / 무형자산의 취득 (25기간, 후보 2개)
  ✅ receivables            [id] ifrs-full_CurrentTradeReceivables / 매출채권 (8기간)
  ✅ inventories            [id] ifrs-full_Inventories / 재고자산 (25기간, 후보 2개)
  ✅ prepaid_expenses       [nm:^선급비용] ifrs-full_CurrentPrepaidExpenses / 선급비용 (

,,,n_periods,yr_min,yr_max,last_FY_억원
sj_div,account_id,account_nm,,,,


## Cell 4 · 품질 점검 실행

In [4]:
# ═══════════════════════════════════════════════════════════════
#  데이터 품질 검증 — 수집·분기화가 제대로 됐는지 종목별/일괄 점검
#  ① 필드 매핑 성공 여부   ② 누적/3개월 감지 결과
#  ③ 분기화 정합성 (연도별 Q1~Q4 + FY 원본 대조, 음수 분기 탐지)
#  ④ 최근 8분기 주요 항목 눈검사 테이블
# ═══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd

KEY_FLOW_FIELDS  = ["revenue", "operating_income", "pretax_income",
                    "tax_expense", "da_cf", "capex_tangible"]
KEY_STOCK_FIELDS = ["receivables", "inventories", "payables",
                    "total_equity", "total_assets", "cash"]
CHECK_FIELDS = KEY_FLOW_FIELDS + KEY_STOCK_FIELDS


def _quarterize_report(ticker: str, field: str = "revenue"):
    """
    flow 필드 1개의 분기화 과정을 연도별 표로 보여준다.
    원본 (Q1,H1,Q3,FY raw) + 감지 모드 + 분기화 결과 (q1~q4) + 정합성 플래그
    """
    raw = _load_ticker_long(ticker, DB_INFO, TABLE_DART_FS)
    spec = FIELD_MAP[field]
    sel = _match_field(raw, spec)
    if sel.empty:
        print(f"  ({field}: 매칭 계정 없음)")
        return None

    pivot = sel.pivot_table(index="bsns_year", columns="quarter",
                            values="thstrm_amount", aggfunc="first")
    force = True if spec["sj"] == ["CF"] else None
    qvals, cum = _flow_to_quarterly(sel, force_cumulative=force)

    rows = []
    for y in sorted(pivot.index):
        r = {"year": int(y)}
        for q in ("Q1", "H1", "Q3", "FY"):
            r[f"raw_{q}"] = pivot.loc[y].get(q, np.nan)
        for q in ("Q1", "Q2", "Q3", "Q4"):
            r[q.lower()] = qvals.get((y, q), np.nan)
        qs = [r["q1"], r["q2"], r["q3"], r["q4"]]
        r["q_sum"] = np.nansum(qs) if any(pd.notnull(v) for v in qs) else np.nan
        fy = r["raw_FY"]
        r["neg_q"] = int(sum(1 for v in qs if pd.notnull(v) and v < 0))
        r["sum≈FY"] = ("OK" if pd.notnull(fy) and pd.notnull(r["q_sum"])
                       and abs(r["q_sum"] - fy) <= abs(fy) * 0.001 else
                       ("-" if pd.isnull(fy) else "MISMATCH"))
        rows.append(r)

    df = pd.DataFrame(rows).set_index("year")
    print(f"  [{field}] 감지 모드: {'누적(차분 적용)' if cum else '3개월치(Q4=FY−3분기합)'}")
    num_cols = [c for c in df.columns if c not in ("neg_q", "sum≈FY")]
    disp = (df[num_cols] / 1e8).round(0)   # 억원
    disp["neg_q"] = df["neg_q"]
    disp["sum≈FY"] = df["sum≈FY"]
    print(disp.to_string())
    n_neg = int(df["neg_q"].sum())
    n_mis = int((df["sum≈FY"] == "MISMATCH").sum())
    if n_neg:
        print(f"  ⚠️ 음수 분기 {n_neg}개 — 누적/3개월 오감지 또는 정정공시 혼입 의심")
    if n_mis:
        print(f"  ⚠️ 분기합≠FY 연도 {n_mis}개")
    return dict(cumulative=cum, neg_quarters=n_neg, fy_mismatch=n_mis)


def check_ticker(ticker: str, show_quarterize=("revenue", "operating_income", "da_cf")):
    """단일 종목 종합 점검 리포트."""
    tk = norm_ticker(ticker)
    print("=" * 78)
    print(f"■ {tk} 데이터 품질 점검")
    print("=" * 78)

    wide = load_dart_financials_wide(tk, DB_INFO, TABLE_DART_FS, verbose=True)
    if wide.empty:
        print("❌ 데이터 없음")
        return {"ticker": tk, "status": "NO_DATA"}

    # ① 필드 커버리지
    print("\n[① 핵심 필드 커버리지] (비결측 분기 수 / 전체", len(wide), "분기)")
    stat = {}
    for f in CHECK_FIELDS:
        n = int(wide[f].notna().sum()) if f in wide.columns else 0
        stat[f] = n
        mark = "✅" if n >= 12 else ("⚠️" if n > 0 else "❌")
        print(f"  {mark} {f:<20} {n}")

    # ②③ 분기화 상세 (지정 flow 필드)
    print("\n[② 분기화 정합성] (단위: 억원)")
    qz = {}
    for f in show_quarterize:
        res = _quarterize_report(tk, f)
        if res:
            qz[f] = res
        print()

    # ④ 최근 8분기 눈검사
    print("[③ 최근 8분기 주요 항목 (조원)]")
    key = [c for c in CHECK_FIELDS if c in wide.columns]
    print((wide[key].tail(8) / 1e12).round(3).to_string())

    return {"ticker": tk, "status": "OK",
            "n_quarters": len(wide),
            **{f"n_{f}": stat.get(f, 0) for f in CHECK_FIELDS},
            "rev_neg_q": qz.get("revenue", {}).get("neg_quarters", np.nan),
            "rev_cumulative_detected": qz.get("revenue", {}).get("cumulative", None)}


def check_batch(tickers):
    """여러 종목 요약 매트릭스 — '수집이 잘 되는가'를 한 눈에."""
    rows = []
    for tk in tickers:
        try:
            rows.append(check_ticker(tk))
        except Exception as e:
            print(f"❌ {tk}: {e}")
            rows.append({"ticker": norm_ticker(tk), "status": f"ERROR: {str(e)[:60]}"})
        print()
    summary = pd.DataFrame(rows)
    print("=" * 78)
    print("■ 일괄 점검 요약")
    print("=" * 78)
    with pd.option_context("display.width", 220, "display.max_columns", 30):
        print(summary.to_string(index=False))
    return summary


# ── 실행 ──
batch_summary = check_batch(CHECK_TICKERS)


■ 005930 데이터 품질 점검
[22:54:51][005930] wide shape=(44, 22) 기간=2015-03-31~2025-12-31  누적차분 적용: ['capex_tangible', 'capex_intangible']


C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(



[① 핵심 필드 커버리지] (비결측 분기 수 / 전체 44 분기)
  ✅ revenue              24
  ✅ operating_income     39
  ✅ pretax_income        24
  ✅ tax_expense          24
  ❌ da_cf                0
  ✅ capex_tangible       24
  ⚠️ receivables          8
  ✅ inventories          25
  ⚠️ payables             8
  ✅ total_equity         16
  ✅ total_assets         25
  ✅ cash                 25

[② 분기화 정합성] (단위: 억원)
  [revenue] 감지 모드: 3개월치(Q4=FY−3분기합)
        raw_Q1    raw_H1    raw_Q3     raw_FY        q1        q2        q3        q4      q_sum  neg_q    sum≈FY
year                                                                                                             
2019       NaN       NaN  620035.0  2304009.0       NaN       NaN  620035.0       NaN   620035.0      0  MISMATCH
2020  553252.0  529661.0  669642.0  2368070.0  553252.0  529661.0  669642.0  615515.0  2368070.0      0        OK
2021  653885.0  636716.0  739792.0  2796048.0  653885.0  636716.0  739792.0  765655.0  2796048.0      0        OK

C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


  (da_cf: 매칭 계정 없음)

[③ 최근 8분기 주요 항목 (조원)]
            revenue  operating_income  pretax_income  tax_expense  capex_tangible  receivables  inventories  payables  total_equity  total_assets    cash
2024-03-31   71.916             6.606          7.707        0.952          13.422       41.145       53.348    12.419           NaN       470.900  61.906
2024-06-30   74.068            10.444         11.595        1.754          11.941       43.661       55.567    13.113           NaN       485.758  49.844
2024-09-30   79.099             9.183         10.320        0.220          10.959       44.692       53.357    12.862           NaN       491.307  43.131
2024-12-31   75.788             6.493          7.907        0.153          15.086       43.623       51.755    12.370           NaN       514.532  53.706
2025-03-31   79.141             6.685          9.152        0.929          12.128       44.867       53.220    14.496           NaN       516.377  53.161
2025-06-30   74.566             4

...

[22:54:53][005380] wide shape=(44, 20) 기간=2015-03-31~2025-12-31  누적차분 적용: ['capex_tangible', 'capex_intangible']

[① 핵심 필드 커버리지] (비결측 분기 수 / 전체 44 분기)
  ✅ revenue              24
  ✅ operating_income     30
  ✅ pretax_income        24
  ✅ tax_expense          24
  ❌ da_cf                0
  ✅ capex_tangible       24
  ✅ receivables          17
  ✅ inventories          25
  ⚠️ payables             7
  ✅ total_equity         16
  ✅ total_assets         25
  ✅ cash                 25

[② 분기화 정합성] (단위: 억원)
  [revenue] 감지 모드: 3개월치(Q4=FY−3분기합)
        raw_Q1    raw_H1    raw_Q3     raw_FY        q1        q2        q3        q4      q_sum  neg_q    sum≈FY
year                                                                                                             
2019       NaN       NaN  269689.0  1057464.0       NaN       NaN  269689.0       NaN   269689.0      0  MISMATCH
2020  253194.0  218590.0  275758.0  1039976.0  253194.0  218590.0  275758.0  292434.0  1039976.0      0        OK


C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (eng

  (da_cf: 매칭 계정 없음)

[③ 최근 8분기 주요 항목 (조원)]
            revenue  operating_income  pretax_income  tax_expense  capex_tangible  receivables  inventories  payables  total_equity  total_assets    cash
2024-03-31   40.659             3.557          4.727        1.032           1.857          NaN       18.078       NaN           NaN       295.934  19.669
2024-06-30   45.021             4.279          5.566        1.392           1.965          NaN       19.033       NaN           NaN       306.928  18.146
2024-09-30   42.928             3.581          4.370        1.164           1.802          NaN       18.800       NaN           NaN       306.087  14.992
2024-12-31   46.624             2.822          3.119        0.645           2.437          NaN       19.791       NaN           NaN       339.798  19.015
2025-03-31   44.408             3.634          4.465        1.082           2.085          NaN       20.715       NaN           NaN       343.630  17.998
2025-06-30   48.287             3

C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


  (da_cf: 매칭 계정 없음)

[③ 최근 8분기 주요 항목 (조원)]
            revenue  operating_income  pretax_income  tax_expense  capex_tangible  receivables  inventories  payables  total_equity  total_assets   cash
2024-03-31   11.609             0.265          0.323        0.004          -3.972          NaN        9.666       NaN           NaN        82.116  9.263
2024-06-30   12.300             0.406          0.187        0.154          -3.298          NaN        9.760       NaN           NaN        84.188  7.167
2024-09-30   12.670             0.498          0.378       -0.161          -3.431          NaN        9.482       NaN           NaN        88.795  8.832
2024-12-31   12.337            -0.252         -1.157        0.065          -3.913          NaN        8.847       NaN           NaN        93.858  7.855
2025-03-31   12.171             0.447          0.444        0.162          -4.044          NaN        8.529       NaN           NaN        95.091  6.950
2025-06-30   11.418             0.477  

In [8]:
diagnose_accounts("000660", sj_div="CF")

[000660] sj=CF kw=- — 계정 117개 (최근 FY=2024)
                                                                                                                                            n_periods  yr_min  yr_max  last_FY_억원
sj_div account_id                                                                                                         account_nm                                             
CF     dart_ProceedsFromSalesOfOtherFinancialAssets                                                                       기타금융자산의 감소               35    2015    2025         1.0
       dart_ProceedsFromSalesOfLoansAndReceivables                                                                        기타수취채권의 감소               33    2017    2025       382.0
       dart_PurchaseOfLoansAndReceivables                                                                                 기타수취채권의 증가               33    2017    2025       477.0
       dart_PurchaseOfOtherFinancialAssets                         

C:\Users\82108\AppData\Local\Temp\ipykernel_26648\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


n_periods  \
sj_div account_id                                         account_nm                      
CF     dart_ProceedsFromSalesOfOtherFinancialAssets       기타금융자산의 감소                 35   
       dart_ProceedsFromSalesOfLoansAndReceivables        기타수취채권의 감소                 33   
       dart_PurchaseOfLoansAndReceivables                 기타수취채권의 증가                 33   
       dart_PurchaseOfOtherFinancialAssets                기타금융자산의 증가                 28   
       dart_PurchaseOfShortTermFinancialInstruments       단기금융상품의 증가                 27   
...                                                                                 ...   
       ifrs-full_IncreaseDecreaseInCashAndCashEquivalents 현금및현금성자산의순증가(감소)            1   
       ifrs-full_InterestReceivedClassifiedAsOperating... 이자수취                        1   
       ifrs-full_OtherInflowsOutflowsOfCashClassifiedA... 기타 투자활동으로 인한 현금유출입          1   
       ifrs-full_PaymentsFromChangesInOwnershipInteres... 종속회사 지분 추가취득                1   
       -표준계정코드 미사용-                                       장기투자자산의 취득                  1   

                                                                              yr_min  \
sj_div account_id                                         account_nm                   
CF     dart_ProceedsFromSalesOfOtherFinancialAssets       기타금융자산의 감소            2015   
       dart_ProceedsFromSalesOfLoansAndReceivables        기타수취채권의 감소            2017   
       dart_PurchaseOfLoansAndReceivables                 기타수취채권의 증가            2017   
       dart_PurchaseOfOtherFinancialAssets                기타금융자산의 증가            2015   
       dart_PurchaseOfShortTermFinancialInstruments       단기금융상품의 증가            2016   
...                                                                              ...   
       ifrs-full_IncreaseDecreaseInCashAndCashEquivalents 현금및현금성자산의순증가(감소)      2023   
       ifrs-full_InterestReceivedClassifiedAsOperating... 이자수취                  2023   
       ifrs-full_OtherInflowsOutflowsOfCashClassifiedA... 기타 투자활동으로 인한 현금유출입    2024   
       ifrs-full_PaymentsFromChangesInOwnershipInteres... 종속회사 지분 추가취득          2023   
       -표준계정코드 미사용-                                       장기투자자산의 취득            2025   

                                                                              yr_max  \
sj_div account_id                                         account_nm                   
CF     dart_ProceedsFromSalesOfOtherFinancialAssets       기타금융자산의 감소            2025   
       dart_ProceedsFromSalesOfLoansAndReceivables        기타수취채권의 감소            2025   
       dart_PurchaseOfLoansAndReceivables                 기타수취채권의 증가            2025   
       dart_PurchaseOfOtherFinancialAssets                기타금융자산의 증가            2025   
       dart_PurchaseOfShortTermFinancialInstruments       단기금융상품의 증가            2025   
...                                                                              ...   
       ifrs-full_IncreaseDecreaseInCashAndCashEquivalents 현금및현금성자산의순증가(감소)      2023   
       ifrs-full_InterestReceivedClassifiedAsOperating... 이자수취                  2023   
       ifrs-full_OtherInflowsOutflowsOfCashClassifiedA... 기타 투자활동으로 인한 현금유출입    2024   
       ifrs-full_PaymentsFromChangesInOwnershipInteres... 종속회사 지분 추가취득          2023   
       -표준계정코드 미사용-                                       장기투자자산의 취득            2025   

                                                                              last_FY_억원  
sj_div account_id                                         account_nm                      
CF     dart_ProceedsFromSalesOfOtherFinancialAssets       기타금융자산의 감소                 1.0  
       dart_ProceedsFromSalesOfLoansAndReceivables        기타수취채권의 감소               382.0  
       dart_PurchaseOfLoansAndReceivables                 기타수취채권의 증가               477.0  
       dart_PurchaseOfOtherFinancialAssets                기타금융자산의 증가              1096.0  
       dart_PurchaseOfShortTermFinancialInstruments       